# Nelson-Siegel Yield-Curve Fit — Visual Spot-Check

A quick exploration of `fit_nelson_siegel()`. We fit the level/slope/curvature
factors for every `date` x `currency` pair, then pick **one pair at random** and
overlay:

* the **raw zero-rate points** (the data the curve is fitted to), and
* the **fitted Nelson-Siegel curve** reconstructed from the three estimated
  betas and the fixed decay `tau`.

Re-run the random-selection cell to inspect a different pair.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import polars as pl
import pandas as pd
from nelson_siegel_svensson import NelsonSiegelCurve
from plotnine import (
    ggplot, aes, geom_point, geom_line,
    labs, theme_bw, scale_x_continuous,
)

from src import config
from src.data import loaders
from src.factors.nonstyle.yield_curve import fit_nelson_siegel, _tenor_to_years

## 1. Load the zero curve and fit the factors

In [ ]:
cfg = config.load(PROJECT_ROOT / "config.yaml")
# Anchor the (relative) data root to the project root so the notebook works
# regardless of the directory the kernel is launched from.
cfg.raw["data"]["root"] = str(PROJECT_ROOT / cfg["data"]["root"])
tau = float(cfg["factors"]["nelson_siegel"]["decay_tau"])

zero_curve = loaders.load_table("zero_curve", cfg)
ns = fit_nelson_siegel(zero_curve, cfg)
ns.head()

## 2. Pick a random date-currency pair

`rng` is unseeded, so each re-run surfaces a different curve. Set a seed in
`np.random.default_rng(seed)` if you want a reproducible pick.

In [ ]:
rng = np.random.default_rng()
idx = int(rng.integers(ns.height))
pick = ns.row(idx, named=True)

sel_date, sel_ccy = pick["date"], pick["currency"]
level, slope, curvature = pick["level"], pick["slope"], pick["curvature"]
print(f"{sel_ccy} curve on {sel_date}")
print(f"  level={level:.4f}  slope={slope:.4f}  curvature={curvature:.4f}  (tau={tau})")

## 3. Assemble the raw points and the fitted curve

The raw points are the observed `zero_rate`s at their parsed tenors (in years).
The fitted curve is reconstructed from the estimated betas on a dense tenor grid.

In [ ]:
# Observed zero-rate points for the selected pair.
points = (
    zero_curve
    .filter((pl.col("date") == sel_date) & (pl.col("currency") == sel_ccy))
    .select(
        tenor=_tenor_to_years(pl.col("tenor_description")),
        zero_rate=pl.col("zero_rate"),
    )
    .drop_nulls()
    .sort("tenor")
    .collect()
)

# Fitted Nelson-Siegel curve on a dense grid spanning the observed tenors.
curve = NelsonSiegelCurve(level, slope, curvature, tau)
grid = np.linspace(points["tenor"].min(), points["tenor"].max(), 200)
fitted = pd.DataFrame({"tenor": grid, "zero_rate": curve(grid)})

points_pd = points.to_pandas()
points_pd.head()

## 4. Plot points vs fitted curve

In [ ]:
(
    ggplot(mapping=aes("tenor", "zero_rate"))
    + geom_line(fitted, color="#2c7fb8", size=1)
    + geom_point(points_pd, color="#d95f02", size=2.5)
    + scale_x_continuous(name="Tenor (years)")
    + labs(
        title=f"Nelson-Siegel fit — {sel_ccy} on {sel_date}",
        subtitle=(
            f"level={level:.4f}, slope={slope:.4f}, "
            f"curvature={curvature:.4f}, tau={tau}"
        ),
        y="Zero rate",
    )
    + theme_bw()
)